# ex006_PHT3D_06

In [ ]:
import pandas as pd
from IPython.display import display

comparison_rows = []
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CASE_DIR = Path.cwd()
output = CASE_DIR / "output"
results = np.load(output / "results.npy")
headings = (output / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
times = np.load(output / "results_times.npy")
CASE_DIR = Path.cwd()
INPUT_DIR = CASE_DIR / "input_data"
reference = np.load(INPUT_DIR / "PHT3D_06_results.npy")
observations = np.load(INPUT_DIR / "observations.npy")
save_times = reference["time_days"]
fig = plt.figure(figsize=(8, 5.5))
for i, (name, label, color, upper) in enumerate(
    [("Ca", "Ca", "m", 2), ("T", "Tenside", "r", 5), ("Na", "Na", "b", 10)], 1
):
    actual = np.interp(save_times, times, results[:, headings.index(name), -1] * 1000)
    expected = reference[name] * 1000
    comparison_rows.append(
        {"Variable": name, "RMSE": np.sqrt(np.mean((actual - expected) ** 2)), "Unit": "mmol/L"}
    )
    ax = fig.add_subplot(3, 1, i)
    ax.plot(save_times * 24 * 60, actual, color, label=label)
    ax.plot(save_times * 24 * 60, expected, color + "--", label="PHT3D")
    ax.plot(
        observations["time_minutes"], observations[name], color + "o", label=f"{name} (observed)"
    )
    ax.grid(True)
    ax.set_xlim([0, 300])
    ax.set_ylim([0, upper])
    ax.tick_params(axis="both", labelsize=12)
    ax.set_ylabel(f"{name} (mmol/L)", fontsize=12)
    ax.legend(fontsize=12)
ax.set_xlabel("time (min)", fontsize=12)
plt.show()
comparison = pd.DataFrame(comparison_rows)
comparison = comparison.set_index("Variable")
display(comparison.style.format({"RMSE": "{:.6g}"}).set_uuid("ex006_1"))